# Week 2 Day 2: LangChain - Tools, Chains, Memory & Agents

## Task 1: LangChain setup and core concepts

| Yesterday's raw-Python idea | LangChain equivalent | Responsibility |
| --- | --- | --- |
| Provider SDK client and `send_message` calls | **LLM/chat-model wrapper** such as `ChatGoogleGenerativeAI` or `ChatAnthropic` | Presents a consistent model interface to the chain or agent. |
| `execute_tool(name, arguments)` dispatcher | **Tool** created with `@tool` or `StructuredTool` | Gives a callable operation a name, description, and argument schema. |
| Bounded `for` loop that chooses and runs tools | **`AgentExecutor`** | Runs an agent's tool-use loop and returns the final answer, with limits and callbacks available for control and tracing. |
| `chat` history plus `working_memory` dictionary | **Chat/message history plus memory or state** | Preserves conversation context and execution state across turns; current LangChain applications often make this state explicit. |

LangChain adds standard interfaces around the model, prompts, tools, parsers, and execution state. That makes components composable, but it also means provider-specific configuration, callbacks, message formats, and versioned integrations must be understood when debugging.

In [1]:
import os
from getpass import getpass

import langchain
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

print("LangChain version:", langchain.__version__)
print("Provider adapters imported: ChatAnthropic and ChatGoogleGenerativeAI.")

LangChain version: 1.4.0
Provider adapters imported: ChatAnthropic and ChatGoogleGenerativeAI.


## LCEL: prompt -> model -> parser

LCEL (LangChain Expression Language) uses the pipe operator (`|`) to compose runnables into a sequence. Under the hood, each runnable receives the previous runnable's output, and LangChain manages the standard `invoke` interface and message/value conversions between steps. The result is a single chain that can be invoked, streamed, batched, traced, or embedded inside a larger chain.

In [2]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a concise teaching assistant. Explain concepts with one concrete example.",
        ),
        ("human", "Explain {topic} to a Python developer in three sentences."),
    ]
)

api_key = os.environ.get("GEMINI_API_KEY") or getpass(
    "Enter your Gemini API key (input hidden): "
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key,
    temperature=0.2,
)
chain = prompt | llm | StrOutputParser()
answer = chain.invoke({"topic": "LangChain tools"})
print(answer)

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


LangChain tools are standard Python functions wrapped with extra metadata that allow an LLM agent to interact with the external world. 

For example, decorating a `get_stock_price(ticker: str)` function with `@tool` lets the LLM autonomously decide to run your Python code whenever a user asks for live market data. 

LangChain automatically converts your function's type hints and docstrings into a JSON schema for the LLM's function-calling mechanism and passes the parsed arguments directly to your function.


## Task 2: Define and register tools

LangChain tools are ordinary Python functions wrapped with a name, schema, and description. With the `@tool` decorator, the function's name, type annotations, and docstring become metadata that LangChain exposes to the model in the tool prompt. The docstring should therefore say what the tool does and when to use it; vague descriptions lead to unreliable tool selection.

The first two tools reuse yesterday's calculator and weather ideas. The third tool reads a local JSON data source, so the agent can retrieve a customer record instead of relying only on model knowledge.

In [3]:
import ast
import json
import operator
from pathlib import Path

from langchain_core.tools import tool


_BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}


def _evaluate_expression(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
        value = _evaluate_expression(node.operand)
        return value if isinstance(node.op, ast.UAdd) else -value
    if isinstance(node, ast.BinOp) and type(node.op) in _BINARY_OPERATORS:
        left = _evaluate_expression(node.left)
        right = _evaluate_expression(node.right)
        return _BINARY_OPERATORS[type(node.op)](left, right)
    raise ValueError("Only numeric expressions with +, -, *, /, and ** are allowed.")


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic numeric expression. Use for arithmetic with +, -, *, /, and **."""
    parsed_expression = ast.parse(expression, mode="eval")
    result = _evaluate_expression(parsed_expression.body)
    return f"{expression} = {result}"


@tool
def weather_lookup(city: str, unit: str = "celsius") -> dict:
    """Return stub weather data for a city. Use when the user asks about current weather."""
    temperatures = {"london": 18, "tokyo": 26, "new york": 22}
    base_temperature = temperatures.get(city.lower(), 20)
    if unit == "fahrenheit":
        temperature = round(base_temperature * 9 / 5 + 32, 1)
    elif unit == "celsius":
        temperature = base_temperature
    else:
        raise ValueError("unit must be 'celsius' or 'fahrenheit'")
    return {
        "city": city,
        "temperature": temperature,
        "unit": unit,
        "conditions": "Partly cloudy",
        "source": "stub data for learning",
    }


DATA_FILE = Path.cwd() / "customer_records.json"
if not DATA_FILE.exists():
    DATA_FILE.write_text(
        json.dumps(
            [
                {"customer_id": "C001", "name": "Aisha Khan", "plan": "Pro", "status": "active"},
                {"customer_id": "C002", "name": "Daniel Lee", "plan": "Starter", "status": "active"},
                {"customer_id": "C003", "name": "Maya Patel", "plan": "Pro", "status": "paused"},
            ],
            indent=2,
        ),
        encoding="utf-8",
    )


@tool
def lookup_customer(customer_id: str) -> dict:
    """Read customer_records.json and return one customer by customer_id. Use for customer record lookups."""
    records = json.loads(DATA_FILE.read_text(encoding="utf-8"))
    for record in records:
        if record["customer_id"].lower() == customer_id.lower():
            return record
    return {"error": f"No customer found for {customer_id}"}


registered_tools = [calculator, weather_lookup, lookup_customer]
print("Registered tools:", [registered_tool.name for registered_tool in registered_tools])
for registered_tool in registered_tools:
    print(f"- {registered_tool.name}: {registered_tool.description}")

Registered tools: ['calculator', 'weather_lookup', 'lookup_customer']
- calculator: Evaluate a basic numeric expression. Use for arithmetic with +, -, *, /, and **.
- weather_lookup: Return stub weather data for a city. Use when the user asks about current weather.
- lookup_customer: Read customer_records.json and return one customer by customer_id. Use for customer record lookups.


In [4]:
print(calculator.invoke({"expression": "(12 + 8) / 4"}))
print(weather_lookup.invoke({"city": "London", "unit": "celsius"}))
print(lookup_customer.invoke({"customer_id": "C001"}))

(12 + 8) / 4 = 5.0
{'city': 'London', 'temperature': 18, 'unit': 'celsius', 'conditions': 'Partly cloudy', 'source': 'stub data for learning'}
{'customer_id': 'C001', 'name': 'Muhammad Naveed Mehdi', 'plan': 'Pro', 'status': 'active'}


## Task 3: Build an agent with `create_tool_calling_agent`

`create_tool_calling_agent` connects the chat model to the registered tools, while `AgentExecutor` runs the tool-calling loop. `verbose=True` prints LangChain's execution trace, and `return_intermediate_steps=True` preserves the tool actions and observations for inspection.

The trace follows the same ReAct shape as Day 1:

- **Reason:** the model decides whether a tool is needed and prepares arguments. The model's hidden chain-of-thought is not exposed.
- **Act:** the agent emits a tool call, such as `calculator` or `lookup_customer`.
- **Observe:** LangChain executes the tool and feeds its result back into the agent for the next step.

Compared with the raw-Python log, both versions repeat model decision -> tool execution -> result -> next decision. LangChain hides message-history construction, tool-call parsing, dispatch, and loop plumbing behind `AgentExecutor`; the returned intermediate steps show the public action and observation data, but not private reasoning.

In [5]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import MessagesPlaceholder

agent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Use tools when they provide needed facts. "
            "After gathering the facts, give a concise final answer.",
        ),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_tool_calling_agent(llm, registered_tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=registered_tools,
    verbose=True,
    return_intermediate_steps=True,
    max_iterations=6,
)

agent_question = (
    "Calculate (12 + 8) / 4, look up customer C001, and report both results "
    "along with the customer's plan and status."
)
agent_result = agent_executor.invoke({"input": agent_question})

print("\nAnnotated public trace:")
for step_number, (action, observation) in enumerate(
    agent_result["intermediate_steps"], start=1
):
    print(f"{step_number}. ACT -> {action.tool}({action.tool_input})")
    print(f"   OBSERVE -> {observation}")
print(f"FINAL -> {agent_result['output']}")



> Entering new AgentExecutor chain...


f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `calculator` with `{'expression': '(12 + 8) / 4'}`


(12 + 8) / 4 = 5.0
Invoking: `lookup_customer` with `{'customer_id': 'C001'}`


{'customer_id': 'C001', 'name': 'Muhammad Naveed Mehdi', 'plan': 'Pro', 'status': 'active'}

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Here are the results:\n\n* **Calculation:** `(12 + 8) / 4` = **5**\n* **Customer C001 Details:**\n  * **Name:** Muhammad Naveed Mehdi\n  * **Plan:** Pro\n  * **Status:** Active', 'index': 0, 'extras': {'signature': 'EusCCugCARFNMg955FslipHDS47WFlhTjtI2sZHCGiCWLiNcAUiaiqyHKcfqXSv0sSuQDKgv90pNltS69AGkMHqXVDbUQJ7gOemJ5ZJ5nfxjsKEOyLAkPI4trpoOhsmTVlNliqNbNNXBi1bNBhMbgSl54kFnwqW6ubmz5uT2NxTVItB7mZPlh0hkDQpi3fT0uhBzqcFOwVXpkl7z3XJzI1hDwTLJLXaw+Dbmb5FqHnAp4PuQsyWsrq/VJXQ6qyjZQ+RzXr+QFWmlcs0a7TD868gaZQlzjArxgJhvrYqWcR48N5wALL3PA2nEEKddFL/1XeWLvHQ4UDXeQkzhC60FBa38dqsOc08AFzVc4TFfxe9Nk2cY3OysAT14uVGH7VTb90TlVeTM5yr+miymURcgWIsHuIOb4uTGkq47Y3OGia4Ke8IJItWCM4ucnLu3PiEda+29pyVDxs8020c0Lm9jBMcRakMQhf8GqDEZtCYd'}}]

> Finished chain.

Annotated public trace:
1. ACT -> calculator({'expression': '(12 + 8) / 4'})
   OBSERVE -> (12 + 8) / 4 = 5.0
2. ACT -> lookup_customer({'customer_id': 'C001'})
   OBSERVE -> {'customer_id': 'C001', 'name': 'Muhammad Naveed Mehdi', 'plan': 'Pro

## Task 4: Add conversation memory

`ConversationBufferMemory` stores each user input and agent answer, then supplies them as `chat_history` on the next turn. The agent can therefore resolve references such as "it" and "the other one" without repeating the earlier product names.

In [6]:
from langchain_classic.memory import ConversationBufferMemory

PRODUCT_FILE = Path.cwd() / "product_prices.json"
if not PRODUCT_FILE.exists():
    PRODUCT_FILE.write_text(
        json.dumps(
            [
                {"product": "wireless keyboard", "price": 45.0, "currency": "USD"},
                {"product": "mechanical keyboard", "price": 95.0, "currency": "USD"},
                {"product": "ergonomic mouse", "price": 35.0, "currency": "USD"},
            ],
            indent=2,
        ),
        encoding="utf-8",
    )


@tool
def lookup_product_price(product: str) -> dict:
    """Read product_prices.json and return a product's price. Use for product price questions."""
    products = json.loads(PRODUCT_FILE.read_text(encoding="utf-8"))
    for record in products:
        if record["product"].lower() == product.lower():
            return record
    return {"error": f"No price found for {product}"}


memory_tools = [lookup_product_price]
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="output",
)
memory_agent = create_tool_calling_agent(llm, memory_tools, agent_prompt)
memory_agent_executor = AgentExecutor(
    agent=memory_agent,
    tools=memory_tools,
    memory=memory,
    verbose=True,
    return_intermediate_steps=True,
    max_iterations=4,
)

conversation = [
    "Find the price of the wireless keyboard.",
    "Now compare it to the mechanical keyboard.",
    "Which one should I recommend to a budget-conscious client?",
]

conversation_results = []
for turn_number, user_message in enumerate(conversation, start=1):
    print(f"\n===== TURN {turn_number}: {user_message} =====")
    result = memory_agent_executor.invoke({"input": user_message})
    conversation_results.append(result)
    print(f"ANSWER {turn_number}: {result['output']}")

print("\nStored conversation memory:")
print(memory.load_memory_variables({})["chat_history"])

print("\nMemory check: the final turn was answered after the first two turns were stored.")


===== TURN 1: Find the price of the wireless keyboard. =====


> Entering new AgentExecutor chain...


C:\Users\a5455\AppData\Local\Temp\ipykernel_6568\4136729748.py:29: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `lookup_product_price` with `{'product': 'wireless keyboard'}`


{'product': 'wireless keyboard', 'price': 45.0, 'currency': 'USD'}

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The price of the wireless keyboard is $45.', 'index': 0, 'extras': {'signature': 'ErIBCq8BARFNMg+dJM2q106LQlYpOBz9HADQHHnNYP/QCdAmb5jqO99hb8EX7yi5RMjGVNp/dGme7ytDG8ZZtE8AZVexu8vStuOT3Y/ILT0wDCQ6XnGXRpp3IKRI/WRWwYds2mdmUv7I4DGzLboNmcO7DbwqvEEEyCIHnZBEzZQAhVLD6DRhNAv7Mny0SGB56uITMXTyaRfapEow7GFDya1hYFkIgFEoNuF/R73NjsAF6Kn/qw=='}}]

> Finished chain.
ANSWER 1: [{'type': 'text', 'text': 'The price of the wireless keyboard is $45.', 'index': 0, 'extras': {'signature': 'ErIBCq8BARFNMg+dJM2q106LQlYpOBz9HADQHHnNYP/QCdAmb5jqO99hb8EX7yi5RMjGVNp/dGme7ytDG8ZZtE8AZVexu8vStuOT3Y/ILT0wDCQ6XnGXRpp3IKRI/WRWwYds2mdmUv7I4DGzLboNmcO7DbwqvEEEyCIHnZBEzZQAhVLD6DRhNAv7Mny0SGB56uITMXTyaRfapEow7GFDya1hYFkIgFEoNuF/R73NjsAF6Kn/qw=='}}]

===== TURN 2: Now compare it to the mechanical keyboard. =====


> Entering new AgentExecutor chain...


...


Invoking: `lookup_product_price` with `{'product': 'mechanical keyboard'}`


{'product': 'mechanical keyboard', 'price': 95.0, 'currency': 'USD'}

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The mechanical keyboard is priced at $95, which is $50 more expensive than the wireless keyboard ($45).', 'index': 0, 'extras': {'signature': 'ErkDCrYDARFNMg+W1seSRxpSavrJRtnO15E0RG3w0hsAQr9xmttMRsBOoLOKM28vD//XFmabk6ODJKolfJ6X6LR37Ee3YjrkRMIohKF5yBwH+PNANA+mNIB/43aYKMinJvyqhNvI5/XfkvtLSUA3wCL5KvFZxxoeGyRtxlmezqKRQRxQHGAdVikoUwcCEyG7WbPES46+iMEj8w6jvbMGdEpwYC3bcpuQfBrHhoi7H+LXAlKyX0tcUGAYAc8fspGCRqJ/WByyBS/w09XOdbWivPCofpaPvI5f2TsOMuj09bqd+URYioWBC2396ht5fhLLt91ZCD18P374olVOxxWD8Z5upEsrGNmxCC1dX5smjZcCR77zCmS+l9o5OWAe7FLZsw3YM2LuK0KYbmIpSENQyaSLTltWBQ480EyZJ273WPX6YGOfrGKK1ksmqgFBXBu0hlQ9hCgk6DnjK2Jr64b4LQuNR7CgxumbEi1nGS85Q6AgUsQiaHsaNE6uP1ZhqfOxXrmHIlGyQEWt5BvDliLdw65pPKKT7bmUUen4GE60XFe9Irfqn13ivrbTjEffgW2WXpS/oJZRklgUenOp'}}]

> Finished chain.
ANSWER 2: [{'type': 'text', 'text': 'The mechanical keyboard is priced at $95, which is $50 more expensive than the wireless keyboard ($45).', 'index': 0, 'extras': {'signature': 'ErkDCrYDARFNMg+W1seSRxpSavrJRtnO15

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'You should recommend the **wireless keyboard**. At **$45**, it is less than half the price of the mechanical keyboard ($95), making it the far more economical choice for a budget-conscious client.', 'index': 0, 'extras': {'signature': 'EqwECqkEARFNMg9fYoqCjv8wFjMkffBF3UxAR1euaFAKnbn0ToGzb/QZ9kw/XRjnEr9CoC0JD+Kj730Lkqcfmc6ZFVTE2snvHnL/XbQMn+ZbjQdJZ5aoRCjS2p+rzDTNV4gMTjeqL+m+RQ0FKq89hdupmjC4IyZ1P3mjc13/QpaERWKkmhU+wCnaNU/BjcdQQ5vPJ4hAK91ZX0C+A99WS3M5aCBreDWivZtHnm/ualH5Qok4FGs1myp32NrMebbIGfq1TxKYEOBu8n4+JXw0rindCA5Qr7x3knDDyuiqwYRLLGaIck/yI2MxgLjTz1KhMnS3pBu5jybMgTwxfTrw2c3HGMJO46Xe0TIpI13S6zf8Rp7c+J8/yELTtuyoYxAqKEeSM6jJolMvBUBVDCkrTj5DgTE3ztUcXrrit742tFnQtA1Cm7K8g2m8KQ+LZg0i2R/SWLSD8q4McUmxRtULjLVo7fONX1Lky6nENBeCYUQFMy1BvAN1/NnXIKVwkp5Ytflve+raIoKOKUEn1DJ29f7XqW6r0orYSXgJouDAKLcd9qNuSCcy4RPjHi7mDP2kS1aKTDNNxezxnUvPIJ7vE4eet7b2fro35sly7uPTvxHvPGOl2R4RdwC+DH575Jeif9txtFXAZVcEmycmsunqd6nVcbK+jUI3OfVxeKCz0Emer2rdb2SPcRm0Pec9nbctkaRqfywW2Kx+JCchjz2FJlXxN6Vd8zht8

## Task 5: Structured output and error handling

A structured-output model converts the agent's final text into a validated Pydantic object. This is useful when downstream code needs predictable fields instead of prose.

The flaky inventory tool below fails once on purpose. `StructuredTool.from_function(..., handle_tool_error=...)` converts that exception into an observation the agent can understand. The prompt tells the agent to retry after a tool error, and `max_iterations` prevents an endless recovery loop.

In [7]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool, ToolException


class AgentReport(BaseModel):
    """Validated final response returned by the structured-output formatter."""

    summary: str = Field(description="A concise answer to the user's request.")
    recommendation: str = Field(description="The recommended choice or 'not applicable'.")
    confidence: str = Field(description="High, medium, or low confidence.")


flaky_state = {"attempts": 0}


def _flaky_inventory_lookup(item: str) -> dict:
    """Return inventory for an item, failing once to demonstrate recovery."""
    flaky_state["attempts"] += 1
    if flaky_state["attempts"] == 1:
        raise ToolException("Temporary inventory service failure")
    inventory = {
        "wireless keyboard": {"item": item, "in_stock": True, "quantity": 12},
        "mechanical keyboard": {"item": item, "in_stock": True, "quantity": 4},
    }
    return inventory.get(item.lower(), {"item": item, "in_stock": False, "quantity": 0})


def _handle_inventory_error(error: Exception) -> str:
    return f"Inventory lookup failed: {error}. This is temporary; retry the same lookup once."


inventory_tool = StructuredTool.from_function(
    func=_flaky_inventory_lookup,
    name="inventory_lookup",
    description=(
        "Check current stock for a product. If the tool reports a temporary failure, "
        "retry the lookup once before answering."
    ),
    handle_tool_error=_handle_inventory_error,
)

error_handling_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Use tools for factual inventory information. If a tool reports a temporary "
            "failure, retry it once. Then provide a concise final answer.",
        ),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

error_agent = create_tool_calling_agent(llm, [inventory_tool], error_handling_prompt)
error_executor = AgentExecutor(
    agent=error_agent,
    tools=[inventory_tool],
    verbose=True,
    return_intermediate_steps=True,
    max_iterations=4,
)

error_result = error_executor.invoke(
    {"input": "Check whether the wireless keyboard is in stock and report the quantity."}
)
print("Agent recovered with:", error_result["output"])


def _text_from_agent_output(output) -> str:
    if isinstance(output, str):
        return output
    if isinstance(output, list):
        return "".join(
            part.get("text", "") for part in output if isinstance(part, dict)
        )
    return str(output)


structured_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Convert the agent answer into the requested schema. Use 'not applicable' "
            "for recommendation when the request is not a recommendation.",
        ),
        ("human", "Agent answer:\n{agent_answer}"),
    ]
)
structured_chain = structured_prompt | llm.with_structured_output(AgentReport)
structured_report = structured_chain.invoke(
    {"agent_answer": _text_from_agent_output(error_result["output"])}
)

print("\nValidated structured output:")
print(structured_report.model_dump_json(indent=2))
print("\nRecovery attempts:", flaky_state["attempts"])
print("Error handling configuration: ToolException + handle_tool_error + max_iterations")



> Entering new AgentExecutor chain...


f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `inventory_lookup` with `{'item': 'wireless keyboard'}`


Inventory lookup failed: Temporary inventory service failure. This is temporary; retry the same lookup once.

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `inventory_lookup` with `{'item': 'wireless keyboard'}`


{'item': 'wireless keyboard', 'in_stock': True, 'quantity': 12}

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The wireless keyboard is in stock with a quantity of 12.', 'index': 0, 'extras': {'signature': 'ErYCCrMCARFNMg8WXJBphuc6zukLxciq3ikMGC7Gh6DXrPoZs5+jutxDZaxjfa+JPE7v9Fd82+WTYuWPql2TizWQOP0K5DRopgeKccc3HjiAV960/uL4b9Ra5wxvGVRvdCm3CPJtBC15GeSK7rIrTPRpOUALsQx+Kd1qAci+KJmcMRAZQk5tSTdZibXVK5xEXHC8E+HW5k0aVxbTS3eeIc9PCqfGyd/Y8OTODjYSfMxONzTYKDdbkKkNCCkFpobkleiIgjv46ilM08c5Mmzp/CWZuWWT4qh7Iets4VXt8YwjY1rYb8lBFSJRY/kuvt95vqape5lPJ/AhWy3pX6hWaf7vC2PdVee7lLgP+qtfQ+5UAsBGzISlstV6IfuhCPkaAvdGYiQu/ZZlZHOD0u0rq4Grz8zqd/kbXA=='}}]

> Finished chain.
Agent recovered with: [{'type': 'text', 'text': 'The wireless keyboard is in stock with a quantity of 12.', 'index': 0, 'extras': {'signature': 'ErYCCrMCARFNMg8WXJBphuc6zukLxciq3ikMGC7Gh6DXrPoZs5+jutxDZaxjfa+JPE7v9Fd82+WTYuWPql2TizWQOP0K5DRopgeKccc3HjiAV960/uL4b9Ra5wxvGVRvdCm3CPJtBC15GeSK7rIrTPRpOUALsQx+Kd1qAci+KJmcMRAZQk5tSTdZibXVK5xEXHC8E+HW5k0aVxbTS3eeIc9PCqfGyd/Y8OTODjYSfMxONzTYKDdbkKkNCCkFpobkleiIgjv46ilM08c5Mmzp/CWZuWWT4qh7

f:\Web 3 Geeks Internship\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Validated structured output:
{
  "summary": "The wireless keyboard is in stock with a quantity of 12.",
  "recommendation": "not applicable",
  "confidence": "High"
}

Recovery attempts: 2
Error handling configuration: ToolException + handle_tool_error + max_iterations


In [8]:
flaky_state["attempts"] = 0
print("Handled first failure:", inventory_tool.invoke({"item": "wireless keyboard"}))
print("Successful retry:", inventory_tool.invoke({"item": "wireless keyboard"}))

Handled first failure: Inventory lookup failed: Temporary inventory service failure. This is temporary; retry the same lookup once.
Successful retry: {'item': 'wireless keyboard', 'in_stock': True, 'quantity': 12}
